In [ ]:
import pandas as pd
import numpy as np
from utils import risk_assessment, summary_stats_user, create_temporal_features

train = pd.read_csv('clean_data/cleaned_loans_train.csv')
# .set_index('index')
test = pd.read_csv('clean_data/cleaned_loans_test.csv')
# .set_index('Id')
valid = pd.read_csv('clean_data/cleaned_loans_valid.csv')
# .set_index('index')
train_data = create_temporal_features(risk_assessment(train))
valid_data = create_temporal_features(risk_assessment(valid))
test_data = create_temporal_features(risk_assessment(test))

In [131]:
train_data.FirstTimeHomebuyerFlag.replace({'Y':1, 'N':0}, inplace=True)
train_data.SuperConformingFlag.replace({'Y':1, 'N':0}, inplace=True)
valid_data.FirstTimeHomebuyerFlag.replace({'Y':1, 'N':0}, inplace=True)
valid_data.SuperConformingFlag.replace({'Y':1, 'N':0}, inplace=True) 
test_data.FirstTimeHomebuyerFlag.replace({'Y':1, 'N':0}, inplace=True)
test_data.SuperConformingFlag.replace({'Y':1, 'N':0}, inplace=True)

In [132]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score

In [206]:
bool_col = ['FirstTimeHomebuyerFlag', 'SuperConformingFlag', 'CreditScore_is_outlier', 'OriginalDTI_is_outlier', 'EstimatedLTV_all_MissFlag','HighRiskCredit', 'HighLTV', 'HighDTI', 'HighInterestRate']

cat_col = ['OccupancyStatus', 'PropertyType', 'LoanPurpose', 'ProgramIndicator',
           'PropertyValMethod', 'BalloonIndicator']

num_col = [

'CreditScore',
'MI_Pct',
'NumberOfUnits',
'OriginalCLTV',
'OriginalDTI',
'OriginalUPB',
'OriginalLTV',
'OriginalInterestRate',
'OriginalLoanTerm',
'NumberOfBorrowers',


'0_CurrentActualUPB','1_CurrentActualUPB','2_CurrentActualUPB','3_CurrentActualUPB','4_CurrentActualUPB','5_CurrentActualUPB','6_CurrentActualUPB','7_CurrentActualUPB','8_CurrentActualUPB','9_CurrentActualUPB','10_CurrentActualUPB','11_CurrentActualUPB','12_CurrentActualUPB','13_CurrentActualUPB',
'0_CurrentInterestRate','1_CurrentInterestRate','2_CurrentInterestRate','3_CurrentInterestRate','4_CurrentInterestRate','5_CurrentInterestRate','6_CurrentInterestRate','7_CurrentInterestRate','8_CurrentInterestRate','9_CurrentInterestRate','10_CurrentInterestRate','11_CurrentInterestRate','12_CurrentInterestRate','13_CurrentInterestRate',
'0_CurrentNonInterestBearingUPB','1_CurrentNonInterestBearingUPB','2_CurrentNonInterestBearingUPB','3_CurrentNonInterestBearingUPB','4_CurrentNonInterestBearingUPB','5_CurrentNonInterestBearingUPB','6_CurrentNonInterestBearingUPB','7_CurrentNonInterestBearingUPB','8_CurrentNonInterestBearingUPB','9_CurrentNonInterestBearingUPB','10_CurrentNonInterestBearingUPB','11_CurrentNonInterestBearingUPB','12_CurrentNonInterestBearingUPB','13_CurrentNonInterestBearingUPB',
'0_EstimatedLTV','1_EstimatedLTV','2_EstimatedLTV','3_EstimatedLTV','4_EstimatedLTV','5_EstimatedLTV','6_EstimatedLTV','7_EstimatedLTV','8_EstimatedLTV','9_EstimatedLTV','10_EstimatedLTV','11_EstimatedLTV','12_EstimatedLTV','13_EstimatedLTV',
'0_InterestBearingUPB','1_InterestBearingUPB','2_InterestBearingUPB','3_InterestBearingUPB','4_InterestBearingUPB','5_InterestBearingUPB','6_InterestBearingUPB','7_InterestBearingUPB','8_InterestBearingUPB','9_InterestBearingUPB','10_InterestBearingUPB','11_InterestBearingUPB','12_InterestBearingUPB','13_InterestBearingUPB',
'0_MonthlyReportingPeriod','1_MonthlyReportingPeriod','2_MonthlyReportingPeriod','3_MonthlyReportingPeriod','4_MonthlyReportingPeriod','5_MonthlyReportingPeriod','6_MonthlyReportingPeriod','7_MonthlyReportingPeriod','8_MonthlyReportingPeriod','9_MonthlyReportingPeriod','10_MonthlyReportingPeriod','11_MonthlyReportingPeriod','12_MonthlyReportingPeriod','13_MonthlyReportingPeriod',
'0_RemainingMonthsToLegalMaturity','1_RemainingMonthsToLegalMaturity','2_RemainingMonthsToLegalMaturity','3_RemainingMonthsToLegalMaturity','4_RemainingMonthsToLegalMaturity','5_RemainingMonthsToLegalMaturity','6_RemainingMonthsToLegalMaturity','7_RemainingMonthsToLegalMaturity','8_RemainingMonthsToLegalMaturity','9_RemainingMonthsToLegalMaturity','10_RemainingMonthsToLegalMaturity','11_RemainingMonthsToLegalMaturity','12_RemainingMonthsToLegalMaturity','13_RemainingMonthsToLegalMaturity',

'DebtServiceRatio',
'LTV_DTI_Interaction',
'CreditScore_LTV_Ratio',
'CreditScore_DTI_Ratio',
'CompositeRiskScore',
'avg_repayment_ratio',
'std_repayment_ratio',
'pct_months_late',
'repayment_trend_slope',

'0_LoanAge',
'1_LoanAge',
'2_LoanAge',
'3_LoanAge',
'4_LoanAge',
'5_LoanAge',
'6_LoanAge',
'7_LoanAge',
'8_LoanAge',
'9_LoanAge',
'10_LoanAge',
'11_LoanAge',
'12_LoanAge',
'13_LoanAge',

'UPB_std',
'UPB_trend',
'UPB_range',

'0_1_UPB_Diff',
'1_2_UPB_Diff',
'2_3_UPB_Diff',
'3_4_UPB_Diff',
'4_5_UPB_Diff',
'5_6_UPB_Diff',
'6_7_UPB_Diff',
'7_8_UPB_Diff',
'8_9_UPB_Diff',
'9_10_UPB_Diff',
'10_11_UPB_Diff',
'11_12_UPB_Diff',
'12_13_UPB_Diff',
]

In [207]:
X_train = train_data[cat_col + num_col + bool_col]
X_valid = valid_data[cat_col + num_col + bool_col]
X_test = test_data.drop(columns="Id")

y_valid = valid_data[['index', 'target']]

In [208]:
out = ['index', 'target']
y_valid = valid_data[out]

### Greedy

In [212]:
from scipy.sparse import issparse
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.svm import OneClassSVM
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from category_encoders import BinaryEncoder

def greedy_forward_selection(X_train, X_valid, y_valid, all_features, max_features=25):
    """
    Greedily add features one by one to maximize AP.
    """
    selected = []
    remaining = all_features.copy()
    best_ap = 0
    
    num_pipeline = Pipeline([
        ("scaler", MinMaxScaler())
    ])

    preprocess = ColumnTransformer(
        transformers=[("num", num_pipeline, all_features)],
        sparse_threshold=1.0,
    )
    
    X_train_prep = preprocess.fit_transform(X_train[all_features])
    X_valid_prep = preprocess.transform(X_valid[all_features])
    
    if issparse(X_train_prep):
        X_train_prep = X_train_prep.toarray()
        X_valid_prep = X_valid_prep.toarray()
    
    feature_to_idx = {f: i for i, f in enumerate(all_features)}
    
    print("Starting forward feature selection...\n")
    
    for round_num in range(max_features):
        best_feature = None
        best_round_ap = best_ap
        
        for feature in remaining:
            # Try adding this feature
            test_features = selected + [feature]
            test_indices = [feature_to_idx[f] for f in test_features]
            
            X_train_test = X_train_prep[:, test_indices]
            X_valid_test = X_valid_prep[:, test_indices]
            
            # Train and evaluate
            ocsvm = OneClassSVM(kernel='rbf', nu=0.001, gamma=0.001)
            ocsvm.fit(X_train_test)
            
            scores = ocsvm.decision_function(X_valid_test)
            raw_anom = -scores
            min_v, max_v = np.min(raw_anom), np.max(raw_anom)
            anom_score = (raw_anom - min_v) / (max_v - min_v + 1e-12)
            
            ap = average_precision_score(y_valid["target"], anom_score)
            roc_auc = roc_auc_score(y_valid["target"], anom_score)

            if ap > best_round_ap:
                print(f"AP={ap:.4f}, AUC={roc_auc:.4f}")
                best_round_ap = ap
                best_feature = feature
        
        if best_feature is None:
            print(f"\nNo improvement found. Stopping at {len(selected)} features.")
            break
        
        selected.append(best_feature)
        remaining.remove(best_feature)
        best_ap = best_round_ap
        
        print(f"Round {round_num+1}: Added '{best_feature}' -> AP={best_ap:.4f}")
    
    return selected

# Run forward selection
selected_features = greedy_forward_selection(X_train, X_valid, y_valid, num_col, max_features=15)
print(f"\nFinal selected features: {selected_features}")



Starting forward feature selection...

AP=0.1044, AUC=0.4201
AP=0.1228, AUC=0.4868
AP=0.1253, AUC=0.4960
AP=0.1321, AUC=0.5100
AP=0.1461, AUC=0.5390
AP=0.1568, AUC=0.5749
AP=0.1583, AUC=0.5185
AP=0.1772, AUC=0.5712
AP=0.2185, AUC=0.6491
AP=0.2419, AUC=0.6267
AP=0.2474, AUC=0.5940
AP=0.3067, AUC=0.6510
Round 1: Added 'pct_months_late' -> AP=0.3067
AP=0.3319, AUC=0.6361
AP=0.3692, AUC=0.6832
AP=0.3695, AUC=0.6830
AP=0.3696, AUC=0.6830
Round 2: Added '12_RemainingMonthsToLegalMaturity' -> AP=0.3696
AP=0.3703, AUC=0.6767
AP=0.3800, AUC=0.6816
AP=0.3886, AUC=0.6875
AP=0.3887, AUC=0.6880
AP=0.3888, AUC=0.6880
AP=0.3890, AUC=0.6878
AP=0.3976, AUC=0.6850
Round 3: Added 'repayment_trend_slope' -> AP=0.3976
AP=0.4056, AUC=0.7011
AP=0.4121, AUC=0.7021
Round 4: Added 'CreditScore_DTI_Ratio' -> AP=0.4121
AP=0.4204, AUC=0.7066
AP=0.4228, AUC=0.7184
AP=0.4282, AUC=0.7139
Round 5: Added '3_4_UPB_Diff' -> AP=0.4282
AP=0.4383, AUC=0.7184
AP=0.4388, AUC=0.7241
Round 6: Added '4_5_UPB_Diff' -> AP=0.4388
A

In [ ]:
bool_col = []
cat_col = []
num_col = ['pct_months_late', '12_RemainingMonthsToLegalMaturity', 'repayment_trend_slope', 'CreditScore_DTI_Ratio', '3_4_UPB_Diff', '4_5_UPB_Diff', '13_CurrentNonInterestBearingUPB', 'UPB_std']

In [220]:
X_train = train_data[cat_col + num_col + bool_col]
X_valid = valid_data[cat_col + num_col + bool_col]
X_test = test_data.drop(columns="Id")

y_valid = valid_data[['index', 'target']]
out = ['index', 'target']
y_valid = valid_data[out]

In [221]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from category_encoders import BinaryEncoder
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.svm import OneClassSVM


# ============================================================================
# Preprocessing Pipeline
# ============================================================================
num_pipeline = Pipeline([
    ("scaler", MinMaxScaler())
])


preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_col),
        ("cat", BinaryEncoder(handle_unknown="ignore"), cat_col),
        ("bool", "passthrough", bool_col),
    ],
    sparse_threshold=1.0,
)

# ============================================================================
# One-Class SVM Model
# ============================================================================

ocsvm = OneClassSVM(
    kernel='rbf',
    nu=0.0001,  # Expected outlier fraction - tune this!
    gamma='auto',  # RBF kernel parameter
)

# Build pipeline
pipe = Pipeline([
    ("prep", preprocess), 
    ("clf", ocsvm)
])

# ============================================================================
# Training (on normal data only)
# ============================================================================
print("Training One-Class SVM on normal data...")
pipe.fit(X_train)

# ============================================================================
# Validation (for hyperparameter tuning)
# ============================================================================
print("\nEvaluating on validation set...")

# Get anomaly scores
scores = pipe["clf"].decision_function(pipe["prep"].transform(X_valid))

# Convert to [0, 1] range where 1 = anomaly
# One-Class SVM: positive scores = normal, negative = anomaly
# We negate and normalize
raw_anom = -scores  # Now higher = more anomalous

# Min-max normalization to [0, 1]
min_v, max_v = np.min(raw_anom), np.max(raw_anom)
anom_score = (raw_anom - min_v) / (max_v - min_v + 1e-12)

# Store scores
y_valid = y_valid.copy()
y_valid["anom_score"] = anom_score

# Compute metrics
ap = average_precision_score(y_valid["target"], y_valid["anom_score"])
roc_auc = roc_auc_score(y_valid["target"], y_valid["anom_score"])

print(f"Average Precision: {ap:.4f}")
print(f"AUC-ROC: {roc_auc:.4f}")

# ============================================================================
# Hyperparameter Tuning on Validation Set
# ============================================================================
def tune_hyperparameters(X_train, X_valid, y_valid, preprocess):
    """
    Grid search for optimal hyperparameters.
    Focus on nu and gamma which are most critical.
    """
    best_ap = 0
    best_params = None
    best_scores = None
    
    # Hyperparameter grid
    nu_list = [0.0001, 0.0003,0.0005, 0.001, 0.005, 0.01]
    gamma_list = ['scale', 'auto', 0.001, 0.01, 0.1]
    
    print("Starting hyperparameter tuning...\n")
    
    for nu in nu_list:
        for gamma in gamma_list:
            # Create model
            ocsvm = OneClassSVM(
                kernel='rbf',
                nu=nu,
                gamma=gamma,
            )
            
            pipe = Pipeline([("prep", preprocess), ("clf", ocsvm)])
            
            # Train and evaluate
            pipe.fit(X_train)
            scores = pipe["clf"].decision_function(pipe["prep"].transform(X_valid))
            
            raw_anom = -scores
            min_v, max_v = np.min(raw_anom), np.max(raw_anom)
            anom_score = (raw_anom - min_v) / (max_v - min_v + 1e-12)
            
            ap = average_precision_score(y_valid["target"], anom_score)
            roc_auc = roc_auc_score(y_valid["target"], anom_score)
            
            print(f"nu={nu:.3f}, gamma={gamma} -> AP={ap:.4f}, AUC={roc_auc:.4f}")
            
            if ap > best_ap:
                best_ap = ap
                best_params = {'nu': nu, 'gamma': gamma}
                best_scores = {'ap': ap, 'auc': roc_auc}
    
    print(f"\nBest parameters: {best_params}")
    print(f"Best AP: {best_scores['ap']:.4f}, AUC: {best_scores['auc']:.4f}")
    return best_params

# Uncomment to run tuning:
best_params = tune_hyperparameters(X_train, X_valid, y_valid, preprocess)

# ============================================================================
# Retrain with best parameters (after tuning)
# ============================================================================
# After tuning, use best parameters:
print(best_params)
ocsvm = OneClassSVM(kernel='rbf', nu=best_params['nu'], gamma=best_params['gamma'])
pipe = Pipeline([("prep", preprocess), ("clf", ocsvm)])
pipe.fit(X_train)


Training One-Class SVM on normal data...

Evaluating on validation set...
Average Precision: 0.4382
AUC-ROC: 0.7291
Starting hyperparameter tuning...

nu=0.000, gamma=scale -> AP=0.2955, AUC=0.6467
nu=0.000, gamma=auto -> AP=0.4382, AUC=0.7291
nu=0.000, gamma=0.001 -> AP=0.3540, AUC=0.6970
nu=0.000, gamma=0.01 -> AP=0.4406, AUC=0.7340
nu=0.000, gamma=0.1 -> AP=0.4393, AUC=0.7304
nu=0.000, gamma=scale -> AP=0.2954, AUC=0.6468
nu=0.000, gamma=auto -> AP=0.4235, AUC=0.7182
nu=0.000, gamma=0.001 -> AP=0.4387, AUC=0.7264
nu=0.000, gamma=0.01 -> AP=0.4339, AUC=0.7247
nu=0.000, gamma=0.1 -> AP=0.4261, AUC=0.7199
nu=0.001, gamma=scale -> AP=0.3013, AUC=0.6512
nu=0.001, gamma=auto -> AP=0.4215, AUC=0.7106
nu=0.001, gamma=0.001 -> AP=0.4267, AUC=0.7158
nu=0.001, gamma=0.01 -> AP=0.4300, AUC=0.7164
nu=0.001, gamma=0.1 -> AP=0.4235, AUC=0.7116
nu=0.001, gamma=scale -> AP=0.3626, AUC=0.6842
nu=0.001, gamma=auto -> AP=0.4316, AUC=0.7133
nu=0.001, gamma=0.001 -> AP=0.4480, AUC=0.7248
nu=0.001, gamma=

Pipeline(steps=[('prep',
                 ColumnTransformer(sparse_threshold=1.0,
                                   transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   MinMaxScaler())]),
                                                  ['pct_months_late',
                                                   '12_RemainingMonthsToLegalMaturity',
                                                   'repayment_trend_slope',
                                                   'CreditScore_DTI_Ratio',
                                                   '3_4_UPB_Diff',
                                                   '4_5_UPB_Diff',
                                                   '13_CurrentNonInterestBearingUPB',
                                                   'UPB_std']),
                                                 ('cat',
                                                  BinaryEncoder(handle_unknown='ignore'),
                                                  []),
                                                 ('bool', 'passthrough', [])])),
                ('clf', OneClassSVM(gamma=0.001, nu=0.001))])

In [222]:
# ============================================================================
# Test Set Predictions
# ============================================================================
print("\nGenerating test predictions...")

# Transform test data and get scores
test_scores = pipe["clf"].decision_function(pipe["prep"].transform(X_test))

# Convert to [0, 1] anomaly scores
raw_anom_test = -test_scores
min_v, max_v = np.min(raw_anom_test), np.max(raw_anom_test)
test_anom_score = (raw_anom_test - min_v) / (max_v - min_v + 1e-12)

# Create submission
submission = pd.DataFrame({
    'Id': range(len(test_anom_score)),
    'target': test_anom_score
})

submission.to_csv('kait_greedy_features.csv', index=False)
print("Submission saved to 'kait_greedy_features.csv'")
print(f"\nSample predictions:\n{submission.head(10)}")
print(f"Score statistics: min={test_anom_score.min():.4f}, max={test_anom_score.max():.4f}, mean={test_anom_score.mean():.4f}")


Generating test predictions...
Submission saved to 'kait_greedy_features.csv'

Sample predictions:
   Id    target
0   0  0.000364
1   1  0.000325
2   2  0.000313
3   3  0.000337
4   4  0.000274
5   5  0.000326
6   6  0.000297
7   7  0.000328
8   8  0.000358
9   9  0.000363
Score statistics: min=0.0000, max=1.0000, mean=0.0036


In [223]:
submission

,Id,target
0,0,0.000364
1,1,0.000325
2,2,0.000313
3,3,0.000337
4,4,0.000274
...,...,...
13421,13421,0.000329
13422,13422,0.000316
13423,13423,0.000364
13424,13424,0.000370


In [224]:
y_pred = pipe["clf"].predict(pipe["prep"].transform(X_valid))
y_pred_binary = (y_pred == -1).astype(int)  # Convert to 1 for anomaly, 0 for normal


In [225]:
y_pred_binary

array([0, 0, 0, ..., 0, 0, 0])

In [226]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_valid.target, y_pred_binary))
print("Precision:", precision_score(y_valid.target, y_pred_binary))
print("Recall:", recall_score(y_valid.target, y_pred_binary))
print("F1 Score:", f1_score(y_valid.target, y_pred_binary))
print("ROC-AUC:", roc_auc_score(y_valid.target, y_pred_binary))
print("\nConfusion Matrix:\n", confusion_matrix(y_valid.target, y_pred_binary))
print("\nClassification Report:\n", classification_report(y_valid.target, y_pred_binary))


Accuracy: 0.8888268156424581
Precision: 0.925531914893617
Recall: 0.12850812407680945
F1 Score: 0.22568093385214008
ROC-AUC: 0.5635082704338873

Confusion Matrix:
 [[4686    7]
 [ 590   87]]

Classification Report:
               precision    recall  f1-score   support

         0.0       0.89      1.00      0.94      4693
         1.0       0.93      0.13      0.23       677

    accuracy                           0.89      5370
   macro avg       0.91      0.56      0.58      5370
weighted avg       0.89      0.89      0.85      5370

